In [1]:
import autograd.numpy as np
from autograd import grad
import matplotlib.pyplot as plt
import pickle
import torch
import torch.nn as nn
import time
rs = np.random.RandomState(seed= 2000) # Earlier seed = 10000

In [2]:
def get_mag(cfgs: np.ndarray):
    """Return mean and error of magnetization."""
    axis = tuple([i+1 for i in range(len(cfgs.shape)-1)])
    return jackknife(cfgs.mean(axis=axis))

def get_abs_mag(cfgs: np.ndarray):
    """Return mean and error of absolute magnetization."""
    axis = tuple([i+1 for i in range(len(cfgs.shape)-1)])
    return jackknife(np.abs(cfgs.mean(axis=axis)))

def get_chi2(cfgs: np.ndarray):
    """Return mean and error of suceptibility."""
    V = np.prod(cfgs.shape[1:])
    axis = tuple([i+1 for i in range(len(cfgs.shape)-1)])
    mags = cfgs.mean(axis=axis)
    return jackknife(V * (mags**2 - mags.mean()**2))

def get_corr_func(cfgs: np.ndarray):
    """Return connected two-point correlation function with errors for symmetric lattices."""
    mag_sq = np.mean(cfgs)**2
    corr_func = []
    axis = tuple([i+1 for i in range(len(cfgs.shape)-1)])

    for i in range(1, cfgs.shape[1], 1):
        corrs = []

        for mu in range(len(cfgs.shape)-1):
            corrs.append(np.mean(cfgs * np.roll(cfgs, i, mu+1), axis=axis))

        corrs = np.array(corrs).mean(axis=0)
        corr_mean, corr_err = jackknife(corrs - mag_sq)
        corr_func.append([i, corr_mean, corr_err])

    return np.array(corr_func)

In [3]:
# n_therm = 512
# cfgs = np.stack(list(map(grab, phi4_ens['x'])), axis=0)[n_therm:]
# C = 0
# for x in range(L):
# for y in range(L):
# C = C + cfgs*np.roll(cfgs, (-x, -y), axis=(1,2))
# X = np.mean(C, axis=(1,2))
# def bootstrap(x, *, Nboot, binsize):
#     boots = []
#     x = x.reshape(-1, binsize, *x.shape[1:])
#     for i in range(Nboot):
#         boots.append(np.mean(x[np.random.randint(len(x), size=len(x))], axis=(0,1)))
#     return np.mean(boots), np.std(boots)
# X_mean, X_err = bootstrap(X, Nboot=100, binsize=4)

# print(f'Two-point susceptibility = {X_mean:.2f} +/- {X_err:.2f}')
# print(f'... vs HMC estimate = 0.75 +/- 0.01')

In [4]:
def hmc(logp, n_samples,burn_in, x0, n_steps, step_size ,m2 , lamda):
    """Run Hamiltonian Monte Carlo to draw `n_samples` from the log density
    `logp`, starting at initial state `x0`.
    """
    momenta_dist = rs.randn

    # Kinetic and potential energy functions.
    T = lambda r: 0.5*np.sum(r**2)
    V = lambda x: -logp(x,m2,lamda)

    grad_V = grad(V)
    dim = np.shape(x0)
    samples = np.empty((n_samples + burn_in,*dim))
    samples[0] = x0
    cfgs = []
    accept = []
    for i in range(1, n_samples+burn_in):

        x_curr = samples[i-1]
        r_curr = momenta_dist(*dim)
        x_prop, r_prop = leapfrog(x_curr, r_curr, n_steps, step_size, grad_V)

        H_prop = T(r_prop) + V(x_prop)
        H_curr = T(r_curr) + V(x_curr)
        alpha  = np.exp(-H_prop + H_curr)

        if rs.rand() < alpha:
            x_curr = x_prop 
            accept.append(1)
        else:
            accept.append(0)
        
        samples[i] = x_curr
        
        if i>=burn_in:
            cfgs.append(x_curr)

    return np.array(cfgs) , accept 

In [5]:
def leapfrog(x, r, n_steps, step_size, grad_V):
    """Run the leapfrog integrator forward `n_steps` using step size
    `step_size`.
    """
    x, r = x[:], r[:]
    #print(grad_V(x).shape)
    for _ in range(n_steps):
        r = r - (step_size / 2) * grad_V(x)
        x = x + step_size * r
        r = r - (step_size / 2) * grad_V(x)
    return x, r

In [6]:
# 16x16:   56.75 * x0^2 - 6.431 * x0 * x1 - 6.431 * x0 * x2
# 8x8:   199.92 * x0^2 - 13.22 * x0 * x1 - 13.22 * x0 * x2

In [7]:
def logp(x,m2 = 1,lamda = 4.0.):
    potential = m2*x**2 + lamda*x**4
    
    for mu in range(2):
        potential -= x*np.roll(x,-1,mu)
        potential -= x*np.roll(x,1,mu)
    return -np.sum(potential)

In [8]:
def magnetisation(cfgs):
    mag = np.mean(cfgs,axis = (1,2))
    mag = mag**2
    return np.mean(mag)

In [9]:
n_samples = 100000
burn_in = 5000
# lamda_set = np.linspace(3.5,7.5,41)
no_accepted_samples = []
(l,l) = (8,8) # lattice Size
lamda = 4.0
m2 = 1.0
print(lamda)
start = time.time()
cfgs , accept  = hmc(logp= logp,n_samples=n_samples,burn_in = burn_in,x0=rs.randn(l,l),n_steps=50,step_size=0.025,m2 = m2 ,lamda = lamda)
lattices = cfgs
print(f"lamda:{lamda}, AR: {np.mean(accept)*100}")


stop = time.time()        
output = open('data/RG_phi4_8x8_lattices_generated_samples__.pkl', 'wb')
pickle.dump(lattices, output)
output.close()
print("time :",stop-start)

0.0
lamda:0.0, AR: 86.72273069267327
time : 7085.25163936615


In [ ]:
# lattices.shape

In [ ]:
# neg_logp = - logp(lattices,m2 = 1.0,lamda = 0.5)

In [ ]:
# array = [lattices , neg_logp]
# output1 = open('data/phi4_12x12_lattices_for_lamda_4_mass_4_with_neglogp.pkl', 'wb')
# pickle.dump(array, output1)
# output.close()

In [ ]:
lat = torch.tensor(lattices).reshape((-1,1,8,8))

In [ ]:
lat.shape

In [ ]:
avg_pool = nn.AvgPool2d(kernel_size=2)


In [ ]:
coarsed_lattices = avg_pool(lat)

In [ ]:
coarsed_lattices.shape

In [ ]:
output_new = open('data/phi4_32x32_coarsed_lattices_for_lamda_point5_mass_1.pkl', 'wb')
pickle.dump(coarsed_lattices, output_new)
output_new.close()
